# Low Birth Weight Prediction — Logistic Regression\n

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# Cell 1 — Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict
)
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score,
    precision_score, recall_score, accuracy_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, make_scorer
)
from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, PolynomialFeatures
)
from sklearn.calibration import CalibratedClassifierCV

# Resampling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# Models
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Hyperparameter optimisation
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

SEED = 42
np.random.seed(SEED)

print("✅ All libraries imported successfully.")


In [ ]:
# ============================================================
# Cell 2 — Load Dataset
# ============================================================
path = PROJECT_ROOT / "data" / "low_birth_weight" / "raw" / "baby-weights-dataset.csv"
df = pd.read_csv(path)

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print("\n--- First 5 rows ---")
display(df.head())
print("\n--- Data types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")
print("\n--- Descriptive Statistics ---")
df.describe()


In [ ]:
# ============================================================
# Cell 3 — Create Target Variable
# ============================================================
df['low_birth_weight'] = (df['BWEIGHT'] < 5.5).astype(int)
print("--- Target distribution ---")
print(df['low_birth_weight'].value_counts())
print(f"\nImbalance ratio: {df['low_birth_weight'].value_counts()[0] / df['low_birth_weight'].value_counts()[1]:.4f} : 1")

# Visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['low_birth_weight'].value_counts().plot.bar(ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution (count)')
axes[0].set_xticklabels(['Normal (0)', 'Low BW (1)'], rotation=0)
axes[0].set_ylabel('Count')

df['low_birth_weight'].value_counts(normalize=True).plot.pie(
    ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
    labels=['Normal (0)', 'Low BW (1)'], startangle=90
)
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 4 — Drop Irrelevant / Leakage Columns
# ============================================================
df.drop(columns=['ID', 'BWEIGHT'], inplace=True)
print(f"Shape after dropping ID & BWEIGHT: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")


In [ ]:
# ============================================================
# Cell 5 — Identify column types
# ============================================================
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'low_birth_weight']

medical_conditions = [
    'ANEMIA', 'CARDIAC', 'ACLUNG', 'DIABETES', 'HERPES', 'HYDRAM',
    'HEMOGLOB', 'HYPERCH', 'HYPERPR', 'ECLAMP', 'CERVIX', 'PINFANT',
    'PRETERM', 'RENAL', 'RHSEN', 'UTERINE'
]

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Medical condition flags ({len(medical_conditions)}): {medical_conditions}")


In [ ]:
# ============================================================
# Cell 6 — Handle missing values
# ============================================================
print("Missing values BEFORE imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled {col} NaNs with median = {median_val}")

for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled {col} NaNs with mode = {mode_val}")

print(f"\nTotal missing after imputation: {df.isnull().sum().sum()}")


In [ ]:
# ============================================================
# Cell 7 — Feature Engineering
# ============================================================
# --- (a) Interaction features between key medical conditions ---
df['HYPER_ECLAMP'] = df['HYPERPR'] * df['ECLAMP']       # pregnancy hypertension x eclampsia
df['HYPER_CHRONIC'] = df['HYPERCH'] * df['HYPERPR']     # chronic x pregnancy hypertension
df['DIABETES_HYPER'] = df['DIABETES'] * df['HYPERPR']   # diabetes x pregnancy hypertension
df['ANEMIA_CARDIAC'] = df['ANEMIA'] * df['CARDIAC']     # anemia x cardiac
df['PRETERM_CERVIX'] = df['PRETERM'] * df['CERVIX']     # previous preterm x incompetent cervix
df['UTERINE_HYDRAM'] = df['UTERINE'] * df['HYDRAM']    # uterine bleeding x hydramnios

# --- (b) Aggregate medical risk score ---
df['MEDICAL_RISK_SCORE'] = df[medical_conditions].sum(axis=1)
df['HAS_ANY_CONDITION'] = (df['MEDICAL_RISK_SCORE'] > 0).astype(int)
df['HIGH_RISK'] = (df['MEDICAL_RISK_SCORE'] >= 2).astype(int)  # 2+ conditions

# --- (c) Key numeric interactions ---
df['AGE_DIFF'] = df['FAGE'] - df['MAGE']               # father-mother age gap
df['EDUC_DIFF'] = df['FEDUC'] - df['MEDUC']            # education gap
df['MAGE_SQUARED'] = df['MAGE'] ** 2                    # quadratic maternal age
df['WEEKS_SQUARED'] = df['WEEKS'] ** 2                  # quadratic gestational weeks
df['VISITS_PER_WEEK'] = df['VISITS'] / (df['WEEKS'] + 1)  # visit rate
df['GAINED_PER_WEEK'] = df['GAINED'] / (df['WEEKS'] + 1)  # weight gain rate
df['CIG_DRINK_COMBO'] = df['CIGNUM'] + df['DRINKNUM']  # combined substance use

print(f"Shape after feature engineering: {df.shape}")
print(f"New features added: {df.shape[1] - 36}")
print(f"\nNew columns: {[c for c in df.columns if c not in numeric_cols + categorical_cols + ['low_birth_weight']]}")


In [ ]:
# ============================================================
# Cell 8 — Encode Categorical Columns & Feature Selection
# ============================================================
le_map = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_map[col] = dict(zip(le.classes_, range(len(le.classes_))))
    print(f"  LabelEncoded {col}: {le_map[col]}")

# --- Correlation-based Feature Selection ---
CORR_THRESHOLD = 0.02
target = 'low_birth_weight'
corr = df.select_dtypes(include=[np.number]).corr()[target].drop(target).abs()
drop_cols = corr[corr < CORR_THRESHOLD].index.tolist()
keep_cols = corr[corr >= CORR_THRESHOLD].index.tolist()

print(f"\n--- Feature Selection (|corr| >= {CORR_THRESHOLD}) ---")
print(f"Dropped {len(drop_cols)} features: {drop_cols}")
print(f"Kept {len(keep_cols)} features: {keep_cols}")

df_filtered = df.drop(columns=drop_cols)
df_tree = df_filtered.copy()
# Re-identify remaining categorical columns for one-hot encoding
remaining_cat_cols = [c for c in categorical_cols if c in df_filtered.columns]
if remaining_cat_cols:
    df_linear = pd.get_dummies(df_filtered, columns=remaining_cat_cols, drop_first=True)
else:
    df_linear = df_filtered.copy()

print(f"\nTree-model DataFrame shape: {df_tree.shape}")
print(f"Linear-model DataFrame shape: {df_linear.shape}")


In [ ]:
# ============================================================
# Cell 9 — Define Features and Target
# ============================================================
target = 'low_birth_weight'
X = df_linear.drop(columns=[target])
y = df_linear[target]

# Compute scale_pos_weight for boosting models
neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight = neg_count / pos_count

# Remove constant (zero-variance) features
constant_features = [col for col in X.columns if X[col].nunique() <= 1]

print("Constant features:", constant_features)

# Drop constant features
X = X.drop(columns=constant_features)

print(f"Remaining predictors: {X.shape[1]}")

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")


In [ ]:
# ============================================================
# Cell 10 — Evaluation Helper Functions
# ============================================================
def calculate_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    ba = balanced_accuracy_score(y_true, y_pred)
    return acc, prec, rec, f1, mcc, auc, ba

def print_fold_report(y_true, y_pred, fold_num=1):
    print(f"\n--- Classification Report (Fold {fold_num}) ---")
    print(classification_report(y_true, y_pred, target_names=['Normal Weight', 'Low Weight']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

def find_best_threshold(y_true, y_prob, metric='f1'):
    thresholds = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(y_true, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
    best_idx = np.argmax(scores)
    return thresholds[best_idx], scores[best_idx]

print("✅ Helper functions and plotting utilities defined.")


In [ ]:
# ============================================================
# Cell 11  Optuna: Soft Voting Ensemble Weights
# ============================================================
from sklearn.ensemble import VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import time

def get_base_models():
    lr = LogisticRegression(class_weight='balanced', solver='saga', max_iter=1000, random_state=SEED)
    svm_base = LinearSVC(class_weight='balanced', max_iter=1000, random_state=SEED)
    svm = CalibratedClassifierCV(svm_base, cv=3)
    return [('lr', lr), ('svm', svm)]

print("Pre-calculating probabilities to make Optuna lightning fast...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
oof_probs = {'lr': [], 'svm': []}
y_true_opt = []

# Subsample dataset just for finding weights to save massive amounts of time
X_opt = X.sample(frac=0.3, random_state=SEED)
y_opt = y.loc[X_opt.index]

for train_idx, val_idx in skf.split(X_opt, y_opt):
    X_tr, X_val = X_opt.iloc[train_idx], X_opt.iloc[val_idx]
    y_tr, y_val = y_opt.iloc[train_idx], y_opt.iloc[val_idx]
    
    scaler = StandardScaler()
    X_tr_scaled = pd.DataFrame(scaler.fit_transform(X_tr), columns=X.columns)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X.columns)
    
    smote = SMOTE(random_state=SEED)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr_scaled, y_tr)
    
    models = get_base_models()
    for name, model in models:
        model.fit(X_tr_res, y_tr_res)
        probs = model.predict_proba(X_val_scaled)[:, 1]
        oof_probs[name].extend(probs)
    
    y_true_opt.extend(y_val)

def objective_ensemble(trial):
    w1 = trial.suggest_float('w_lr', 0.1, 1.0)
    w2 = trial.suggest_float('w_svm', 0.1, 1.0)
    
    tot = w1 + w2
    w1, w2 = w1/tot, w2/tot
    
    y_prob = w1 * np.array(oof_probs['lr']) + w2 * np.array(oof_probs['svm'])
    best_t, best_score = find_best_threshold(np.array(y_true_opt), y_prob)
    return best_score

print("Tuning 2-Model Ensemble Weights with Optuna...")
study_ens = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_ens.optimize(objective_ensemble, n_trials=30, show_progress_bar=False)

print(f"\n Best Ensemble F1: {study_ens.best_value:.4f}")
print(f"Best params (weights): {study_ens.best_params}")


In [ ]:
# ============================================================
# Cell 12  Create Final Ensemble Model
# ============================================================
best_w = study_ens.best_params
tot = best_w['w_lr'] + best_w['w_svm']
final_weights = [best_w['w_lr']/tot, best_w['w_svm']/tot]

clf_model = VotingClassifier(estimators=get_base_models(), voting='soft', weights=final_weights)
print("Ensemble Model Created with weights:", final_weights)


In [ ]:
# ============================================================
# Cell 13 — 5-Fold Stratified CV (SMOTE + Scaling on Training Fold, No Data Leakage)
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []
train_fold_metrics = []
ya_sm_list = []          # true labels (all folds)
ba_sm_list = []          # predicted probabilities (all folds)
y_pred_all_list = []     # predicted labels (all folds)

print("Starting 5-Fold Stratified Cross-Validation (Scaling + SMOTE inside folds) ...\n")
print("=" * 80)

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train_fold, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # --- Fit scaler on training fold only (No Data Leakage) ---
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_fold), columns=X.columns)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X.columns)

    # --- Apply standard SMOTE only on the scaled training fold ---
    smote = SMOTE(random_state=SEED)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train_fold)

    # --- Train ---
    from sklearn.base import clone
    model = clone(clf_model)
    model.fit(X_train_res, y_train_res)

    # --- Validation predictions (on scaled unresampled val fold) ---
    y_prob = model.predict_proba(X_val_scaled)[:, 1]
    best_t, _ = find_best_threshold(y_val, y_prob)
    y_pred = (y_prob >= best_t).astype(int)

    # --- Validation metrics ---
    acc, prec, rec, f1, mcc, auc, ba = calculate_metrics(y_val, y_pred, y_prob)

    fold_metrics.append({
        'fold': fold, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'mcc': mcc,
        'roc_auc': auc, 'balanced_accuracy': ba
    })

    # --- Training metrics (on ORIGINAL UNRESAMPLED train fold) ---
    y_train_prob = model.predict_proba(X_train_scaled)[:, 1]
    y_train_pred = (y_train_prob >= best_t).astype(int)

    train_acc  = accuracy_score(y_train_fold, y_train_pred)
    train_prec = precision_score(y_train_fold, y_train_pred, zero_division=0)
    train_rec  = recall_score(y_train_fold, y_train_pred, zero_division=0)
    train_f1   = f1_score(y_train_fold, y_train_pred, zero_division=0)
    train_auc  = roc_auc_score(y_train_fold, y_train_prob)

    train_fold_metrics.append({
        'fold': fold, 'accuracy': train_acc, 'precision': train_prec,
        'recall': train_rec, 'f1': train_f1, 'roc_auc': train_auc
    })

    # --- Store predictions ---
    ya_sm_list.extend(y_val.tolist())
    ba_sm_list.extend(y_prob.tolist())
    y_pred_all_list.extend(y_pred.tolist())

    print(f"         Val  — Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} "
          f"| F1: {f1:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f} | BA: {ba:.4f} (Threshold: {best_t:.2f})")
    print(f"         Train— Acc: {train_acc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} "
          f"| F1: {train_f1:.4f} | AUC: {train_auc:.4f}")
    print("-" * 80)

# --- Aggregate DataFrames ---
df_val_metrics   = pd.DataFrame(fold_metrics)
df_train_metrics = pd.DataFrame(train_fold_metrics)

# --- Aggregated confusion matrix ---
cm_all = confusion_matrix(ya_sm_list, y_pred_all_list)
overall_auc = roc_auc_score(ya_sm_list, ba_sm_list)
print(f"\nOverall AUC (aggregated): {overall_auc:.4f}")
print(f"Aggregated Confusion Matrix:\n{cm_all}")

# --- Final model on full resampled dataset ---
scaler_full = StandardScaler()
X_full_scaled = pd.DataFrame(scaler_full.fit_transform(X), columns=X.columns)
smote_full = SMOTE(random_state=SEED)
X_full_res, y_full_res = smote_full.fit_resample(X_full_scaled, y)
final_model = clone(clf_model)
final_model.fit(X_full_res, y_full_res)
print("\n✅ Final model trained on full scaled resampled dataset.")


In [ ]:
# ============================================================
# Cell 14 — Overfitting Analysis
# ============================================================
compare_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

rows = []
for i in range(len(df_train_metrics)):
    fold_num = i + 1
    for m in compare_metrics:
        t_val = df_train_metrics.loc[i, m]
        v_val = df_val_metrics.loc[i, m]
        rows.append({
            'Fold': fold_num,
            'Metric': m,
            'Train': t_val,
            'Validation': v_val,
            'Gap (Train−Val)': t_val - v_val
        })

df_overfit = pd.DataFrame(rows)
print("=" * 75)
print("OVERFITTING ANALYSIS — Train vs Validation per Fold (Unresampled Train)")
print("=" * 75)
display(df_overfit.pivot_table(
    index='Fold',
    columns='Metric',
    values=['Train', 'Validation', 'Gap (Train\u2212Val)']
).round(4))

# Mean summary
print("\n--- Mean across folds ---")
for m in compare_metrics:
    subset = df_overfit[df_overfit['Metric'] == m]
    print(f"  {m:>12s}  |  Train: {subset['Train'].mean():.4f}  |  "
          f"Val: {subset['Validation'].mean():.4f}  |  "
          f"Gap: {subset['Gap (Train\u2212Val)'].mean():.4f}")

avg_gap = df_overfit['Gap (Train\u2212Val)'].mean()
print(f"\nOverall average gap: {avg_gap:.4f}")

if avg_gap > 0.15:
    verdict = "WARNING: Model shows significant overfitting \u26a0\ufe0f"
elif avg_gap > 0.05:
    verdict = "MODERATE: Some overfitting detected \u26a0\ufe0f"
else:
    verdict = "GOOD: No significant overfitting \u2705"
print(f"\n>>> {verdict}")

# --- Grouped bar chart ---
mean_train = [df_train_metrics[m].mean() for m in compare_metrics]
mean_val   = [df_val_metrics[m].mean()   for m in compare_metrics]
mean_gaps  = [t - v for t, v in zip(mean_train, mean_val)]

x_pos = np.arange(len(compare_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_train = ax.bar(x_pos - width/2, mean_train, width, label='Train', color='#4c72b0')
bars_val   = ax.bar(x_pos + width/2, mean_val,   width, label='Validation', color='#55a868')

# Annotate gap on top of validation bars
for idx, (bt, bv, gap) in enumerate(zip(bars_train, bars_val, mean_gaps)):
    ax.text(bt.get_x() + bt.get_width()/2, bt.get_height() + 0.005,
            f'{bt.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(bv.get_x() + bv.get_width()/2, bv.get_height() + 0.005,
            f'{bv.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    mid_x = (bt.get_x() + bv.get_x() + bv.get_width()) / 2
    ax.text(mid_x, max(bt.get_height(), bv.get_height()) + 0.03,
            f'gap={gap:+.4f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='red')

ax.set_xticks(x_pos)
ax.set_xticklabels([m.upper() for m in compare_metrics], fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Overfitting Check: Training vs Validation Metrics',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
MODEL_TITLE = 'LR + SVM SOFT VOTING ENSEMBLE'
cm_ov = cm_all
y_true_all = ya_sm_list
y_prob_all = ba_sm_list
ov_auc = overall_auc
fold_summary = df_val_metrics.rename(columns={'fold':'Fold', 'accuracy':'Accuracy', 'precision':'Precision', 'recall':'Recall', 'f1':'F1_Score', 'balanced_accuracy':'Balanced_Acc', 'roc_auc':'AUC'})
X_original = X
y_original = y
feature_names = X.columns

best_thresh, _ = find_best_threshold(y_true_all, y_prob_all)
cm_ov = confusion_matrix(y_true_all, (np.array(y_prob_all) >= best_thresh).astype(int))

# Retraining on all data for feature importance visualization
# ============================================================
# Cell 10 — 6-Chart Comprehensive Visualization
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve
import pandas as pd
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'{MODEL_TITLE} - Full data set (SMOTE + SCALER, NO DATA LEAKAGE)', fontsize=14, fontweight='bold')

sns.heatmap(cm_ov, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=['Class 0','Class 1'], yticklabels=['Class 0','Class 1'])
axes[0,0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Predicted'); axes[0,0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
axes[0,1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={ov_auc:.3f})')
axes[0,1].plot([0,1],[0,1],'k--',lw=2,label='Random')
axes[0,1].fill_between(fpr, tpr, alpha=0.2, color='orange')
axes[0,1].set_xlabel('False Positive Rate'); axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0,1].legend(loc='lower right'); axes[0,1].grid(True, alpha=0.3)

pvals, rvals, _ = precision_recall_curve(y_true_all, y_prob_all)
axes[0,2].plot(rvals, pvals, color='green', lw=2)
axes[0,2].fill_between(rvals, pvals, alpha=0.2, color='green')
axes[0,2].set_xlabel('Recall'); axes[0,2].set_ylabel('Precision')
axes[0,2].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[0,2].grid(True, alpha=0.3)

fds = fold_summary['Fold']
axes[1,0].plot(fds, fold_summary['Accuracy'],'o-',label='Accuracy',lw=2,ms=8)
axes[1,0].plot(fds, fold_summary['Precision'],'s-',label='Precision',lw=2,ms=8)
axes[1,0].plot(fds, fold_summary['Recall'],'^-',label='Recall',lw=2,ms=8)
axes[1,0].plot(fds, fold_summary['F1_Score'],'d-',label='F1-Score',lw=2,ms=8)
axes[1,0].set_xlabel('Fold'); axes[1,0].set_ylabel('Score')
axes[1,0].set_title('Per-Fold Performance', fontsize=12, fontweight='bold')
axes[1,0].set_ylim(0,1); axes[1,0].legend(loc='best'); axes[1,0].grid(True, alpha=0.3)

avg_m = {'Accuracy': fold_summary['Accuracy'].mean(),
         'Balanced\nAcc': fold_summary['Balanced_Acc'].mean(),
         'Precision': fold_summary['Precision'].mean(),
         'Recall': fold_summary['Recall'].mean(),
         'F1-Score': fold_summary['F1_Score'].mean(),
         'AUC': fold_summary['AUC'].mean()}
bars = axes[1,1].bar(avg_m.keys(), avg_m.values(), color='steelblue', edgecolor='black')
axes[1,1].set_ylim(0,1); axes[1,1].set_ylabel('Score')
axes[1,1].set_title('Average Metrics Across Folds', fontsize=12, fontweight='bold')
axes[1,1].set_xticklabels(list(avg_m.keys()), rotation=45, ha='right')
for bar, val in zip(bars, avg_m.values()):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.3f}', ha='center', fontweight='bold')
axes[1,1].grid(True, alpha=0.3, axis='y')

print("\n Retraining on all data for feature importance visualization...")
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter

plot_scaler = StandardScaler()
X_full_scaled = plot_scaler.fit_transform(X_original)
cc_before = Counter(y_original)
smote_full = SMOTE(random_state=42, k_neighbors=min(5, int(cc_before[1]) - 1))
X_full_resampled, y_full_resampled = smote_full.fit_resample(X_full_scaled, y_original)

clf = clf_model
clf.fit(X_full_resampled, y_full_resampled)

# Helper function to extract feature importances
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 16 — Summary Statistics
# ============================================================
print("\n" + "=" * 65)
print("       FINAL SUMMARY")
print("=" * 65)

summary_metrics = ['accuracy', 'precision', 'recall', 'f1',
                   'mcc', 'roc_auc', 'balanced_accuracy']

print(f"{'Metric':<22s} {'Mean':>8s}  \u00b1  {'Std':>8s}")
print("-" * 45)
for m in summary_metrics:
    mean_v = df_val_metrics[m].mean()
    std_v  = df_val_metrics[m].std()
    print(f"{m:<22s} {mean_v:>8.4f}  \u00b1  {std_v:>8.4f}")

print(f"\n{'Overall AUC (agg.)':<22s} {overall_auc:>8.4f}")

print("\n--- Aggregated Confusion Matrix ---")
tn, fp, fn, tp = cm_all.ravel()
print(f"  TN = {tn:,}   FP = {fp:,}")
print(f"  FN = {fn:,}   TP = {tp:,}")

print(f"\n>>> Overfitting verdict: {verdict}")
print("=" * 65)


In [ ]:
# ============================================================
# Cell 17 — Save Model Artifacts
# ============================================================
from shared.model_artifacts import save_model_artifacts
from sklearn.pipeline import Pipeline as SkPipeline

# Wrap scaler + final_model into a sklearn Pipeline for artifact saving
pipeline = SkPipeline(steps=[
    ('scaler', scaler_full),
    ('model', final_model),
])

# Best threshold from aggregated OOF predictions
best_thresh, _ = find_best_threshold(ya_sm_list, ba_sm_list)

# Aggregate train metrics (mean across folds)
train_metrics = {
    'accuracy':  float(df_train_metrics['accuracy'].mean()),
    'precision': float(df_train_metrics['precision'].mean()),
    'recall':    float(df_train_metrics['recall'].mean()),
    'f1':        float(df_train_metrics['f1'].mean()),
    'roc_auc':   float(df_train_metrics['roc_auc'].mean()),
}

# Aggregate validation metrics (mean across folds)
y_true_all = np.array(ya_sm_list)
y_prob_all = np.array(ba_sm_list)
y_pred_all = (y_prob_all >= best_thresh).astype(int)

test_metrics = {
    'accuracy':  float(accuracy_score(y_true_all, y_pred_all)),
    'precision': float(precision_score(y_true_all, y_pred_all, zero_division=0)),
    'recall':    float(recall_score(y_true_all, y_pred_all, zero_division=0)),
    'f1':        float(f1_score(y_true_all, y_pred_all, zero_division=0)),
    'roc_auc':   float(roc_auc_score(y_true_all, y_prob_all)),
}

report = classification_report(y_true_all, y_pred_all, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true_all, y_pred_all).tolist()

metrics_payload = {
    "metrics": test_metrics,
    "classification_report": report,
    "confusion_matrix": cm,
}

# Categorical features that survived feature selection
cat_features = [c for c in remaining_cat_cols if c in X.columns]

save_model_artifacts(
    pipeline=pipeline,
    outcome="lbw",
    outcome_id="lbw",
    algorithm="lr_svm_soft_voting",
    label="Logistic Regression + Linear SVM Soft Voting",
    description="Soft Voting ensemble (LR + SVM) for low birth weight prediction with scaling and SMOTE resampling.",
    target_column="low_birth_weight",
    feature_order=X.columns.tolist(),
    categorical_features=cat_features,
    metrics_payload=metrics_payload,
    positive_label="Low Birth Weight (1)",
    negative_label="Normal Weight (0)",
    threshold=best_thresh,
    y_test=y_true_all,
    y_prob=y_prob_all,
    train_metrics=train_metrics,
    project_root=PROJECT_ROOT,
)
